In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
%cd /content/drive/MyDrive/2026-06-02_pet-project_forecasting-customer-churn/forecasting_customer_churn

/content/drive/MyDrive/2026-06-02_pet-project_forecasting-customer-churn/forecasting_customer_churn


In [33]:
!pwd

/content/drive/MyDrive/2026-06-02_pet-project_forecasting-customer-churn/forecasting_customer_churn


In [28]:
import os
from google.colab import userdata

# Получаем токен из секретов Colab
TOKEN = userdata.get('GITHUB_TOKEN')
USER = 'helloqpworld'
REPO = 'forecasting_customer_churnforecasting_customer_churn'

# Формируем безопасный URL для пуша
# repo_url = f"https://{USER}:{TOKEN}@://github.com{USER}/{REPO}.git"

In [36]:
# !git clone https://github.com/helloqpworld/forecasting_customer_churn.git

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/

nothing added to commit but untracked files present (use "git add" to track)


In [ ]:
# Выполняем пуш через системную команду (f-строка передает скрытый токен)
# !git remote set-url origin {repo_url}
!git push origin main

In [ ]:
import os
if not os.path.exists('response_sample.csv'):
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    ! kaggle competitions download -c findata-response
    ! unzip -q findata-response.zip
    !rm -f responce_sample.csv responce_train.csv findata-response.zip

100% 2.75M/2.75M [00:00<00:00, 249MB/s]



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as ex
import plotly.graph_objs as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
import plotly.offline as pyo
pyo.init_notebook_mode()
sns.set_style('darkgrid')
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score as f1
from sklearn.metrics import confusion_matrix

plt.rc('figure',figsize=(18,9))
%pip install -q imbalanced-learn
from imblearn.over_sampling import SMOTE

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
df_train = pd.read_csv('response_train.csv')
df_sample = pd.read_csv('response_sample.csv')

In [ ]:
df_train.shape

(25101, 52)

In [ ]:
df_train.iloc[:, :12].head(1)

,client_id,target,age,job-status,retrirement-status,gender,children,dependants,education,marital-status,industry,title
0,985668,0,41,1,0,1,3,1,Среднее специальное,Состою в браке,Развлечения/Искусство,Работник сферы услуг


In [ ]:
df_train.iloc[:, 12:24].head(1)

,tp-state,tp-foreign,job-type,family-income,personal-income,registration-region,fact-region,postal-region,last-loan-region,region,reg-and-fact-equality,post-and-fact-equality
0,Государственная комп./учреж.,Без участия,Участие в основ. деятельности,от 10000 до 20000 руб.,5600.0,Чувашия,Чувашия,Чувашия,Чувашия,ПРИВОЛЖСКИЙ,1,1


In [ ]:
df_train.iloc[:, 24:41].head(1)

,reg-and-post-equality,reg-fact-and-post-equality,reg-fact-post-and-last-credit-equality,realty,cars,rus-auto,dacha,cottage,garage,land,last-loan-amount,term,first-payment,driving-license,state-pension-fund,residence-time,work-time
0,1,1,1,0,0,0,0,0,0,0,13000.0,12,4000.0,0,1,144,24.0


In [ ]:
df_train.iloc[:, 41:].head(1)

,fact-phone,reg-phone,work-phone,loans,paid-loans,total-payments,total-of-delinquencies,max-delinquency-no,mean-delinquency-amount,max-delinquency-amount,previous-cards
0,0,0,1,1,0,3,0,0,0.0,0.0,NaN


In [ ]:
# НАШЕЛ ВЫБРОС В RESIDENCE-TIME. В PREVIOUS-CARDS МНОГО Nan

# Был выброс в residence-time равный -26, я заменил на 0
df_train.loc[3399, 'residence-time'] = 0

# Почти все значения previous-cards (количество карт с завершенным сроком использования) как Nan, их заменил на 0
df_train['previous-cards'] = df_train['previous-cards'].fillna(0)

# Все значения равны 0, поэтому удалил столбец
del df_train['driving-license']

In [ ]:
df_train.info(memory_usage='deep', show_counts=True, verbose=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25101 entries, 0 to 25100
Columns: 51 entries, client_id to previous-cards
dtypes: float64(7), int64(31), object(13)
memory usage: 39.6 MB


In [ ]:
categorical_cols = df_train.select_dtypes(include=['object', 'category']).columns
numeric_cols = df_train.select_dtypes(include=['int', 'float']).columns
print(f'Категориальные ({len(categorical_cols)}) + Числовые ({len(numeric_cols)}) = Всего ({len(categorical_cols) + len(numeric_cols)})')

Категориальные (13) + Числовые (38) = Всего (51)


In [ ]:
for col in df_train.select_dtypes(include=['object', 'category']).columns:
    df_train[col] = df_train[col].astype('category')

In [ ]:
df_train.info(memory_usage='deep', show_counts=True, verbose=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25101 entries, 0 to 25100
Columns: 51 entries, client_id to previous-cards
dtypes: category(12), float64(7), int64(31), int8(1)
memory usage: 7.6 MB


In [ ]:
print(f"Количество полных дубликатов: {df_train.duplicated().sum()}")
# Подсчет пропущенных значений в процентах по каждой колонке
missing_values = df_train.isnull().sum()
missing_percent = (missing_values / len(df_train)) * 100

# Объединяем в красивую таблицу (выводим только те колонки, где есть пропуски)
missing_df = pd.DataFrame({'Пропуски (шт)': missing_values, 'Пропуски (%)': missing_percent})
missing_df[missing_df['Пропуски (шт)'] > 0].sort_values(by='Пропуски (%)', ascending=False)

Количество полных дубликатов: 0


,Пропуски (шт),Пропуски (%)
work-time,2262,9.011593
industry,2259,8.999641
tp-state,2259,8.999641
title,2259,8.999641
job-type,2259,8.999641
tp-foreign,2255,8.983706
last-loan-region,491,1.956097
region,2,0.007968


In [ ]:
df_train.describe().loc[['count', 'min', '50%', 'max']].T

,count,min,50%,max
client_id,25101.0,985468.0,1000545.0,1015599.0
target,25101.0,0.0,0.0,1.0
age,25101.0,21.0,39.0,67.0
job-status,25101.0,0.0,1.0,1.0
retrirement-status,25101.0,0.0,0.0,1.0
gender,25101.0,0.0,1.0,1.0
children,25101.0,0.0,1.0,10.0
dependants,25101.0,0.0,0.0,7.0
personal-income,25101.0,24.0,12000.0,13000000.0
reg-and-fact-equality,25101.0,0.0,1.0,1.0


In [ ]:
if len(categorical_cols) > 0:
    print(df_train[categorical_cols].describe().T)
else:
    print("\nКатегориальных признаков не найдено.")

                     count unique                            top   freq
education            25101      7            Среднее специальное  10670
marital-status       25101      5                 Состою в браке  15369
industry             22842     31                       Торговля   3958
title                22842     12                     Специалист  11611
tp-state             22842      5               Частная компания  10785
tp-foreign           22846      2                    Без участия  22576
job-type             22842     10  Участие в основ. деятельности  18839
family-income        25101      5         от 10000 до 20000 руб.  11643
registration-region  25101     82             Краснодарский край   1098
fact-region          25101     83             Краснодарский край   1101
postal-region        25101     82             Краснодарский край   1100
last-loan-region     24610     70             Краснодарский край   1300
region               25099     11                          ЮЖНЫЙ

In [ ]:
df_train['target'].value_counts().reset_index().assign(total_sum = lambda x: x['count'].sum(), percentage = lambda x: (x['count'] / x['count'].sum()) * 100).round(2)

,target,count,total_sum,percentage
0,0,22081,25101,87.97
1,1,3020,25101,12.03


In [ ]:
import scipy.stats as stats
from scipy.stats import chi2_contingency

In [ ]:
for col in categorical_cols:
    contingency_table = pd.crosstab(df_train[~df_train.isna().any(axis=1)][col], df_train[~df_train.isna().any(axis=1)]['target'])
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    if p < 0.01:
        print(f"✅ p-value = {p:.4f} признак {col}")
    else:
        print(f"❌ p-value = {p:.4f} признак {col}")

✅ p-value = 0.0000 признак education
✅ p-value = 0.0003 признак marital-status
✅ p-value = 0.0000 признак industry
✅ p-value = 0.0001 признак title
✅ p-value = 0.0000 признак tp-state
❌ p-value = 0.3865 признак tp-foreign
❌ p-value = 0.7658 признак job-type
✅ p-value = 0.0000 признак family-income
✅ p-value = 0.0000 признак registration-region
✅ p-value = 0.0000 признак fact-region
✅ p-value = 0.0000 признак postal-region
✅ p-value = 0.0000 признак last-loan-region
✅ p-value = 0.0000 признак region


In [ ]:
income_mapping = {
    'до 5000 руб.': 2,
    'от 5000 до 10000 руб.': 8,
    'от 10000 до 20000 руб.': 15,
    'от 20000 до 50000 руб.': 35,
    'свыше 50000 руб.': 100
}
df_train['family-income'] = df_train['family-income'].map(income_mapping).astype('int8')

In [ ]:
# cols_for_del = ['registration-region', 'postal-region', 'last-loan-region', 'region', 'max-delinquency-amount',
#                 'paid-loans', 'reg-fact-and-post-equality', 'reg-and-post-equality', 'retrirement-status',
#                 'total-payments', 'client_id', 'job-status', 'tp-foreign', 'job-type', 'previous-cards', 'work-phone',
#                 'post-and-fact-equality', 'cars', 'rus-auto', 'cottage', 'garage', 'land', 'work-time', 'reg-phone',
#                 'age', 'reg-fact-post-and-last-credit-equality', 'industry', 'title', 'tp-state', 'family-income',
#                 'reg-and-fact-equality', 'last-loan-amount', 'first-payment', 'state-pension-fund', 'residence-time',
#                 'total-of-delinquencies', 'personal-income', 'driving-license']

In [ ]:
%pip install -q phik
import phik
from phik import report

In [ ]:
phik_matrix = df_train[~df_train.isna().any(axis=1)].phik_matrix()

interval columns not set, guessing: ['target', 'age', 'retrirement-status', 'gender', 'children', 'dependants', 'family-income', 'reg-and-post-equality', 'last-loan-amount', 'term', 'first-payment', 'residence-time', 'fact-phone', 'paid-loans', 'total-payments', 'total-of-delinquencies', 'max-delinquency-no', 'mean-delinquency-amount', 'previous-cards']


In [ ]:
styled_matrix = phik_matrix.style.background_gradient(cmap='coolwarm', vmin=0, vmax=1).format("{:.4f}")

# Настраиваем стили: выравнивание и уменьшенный размер текста
styled_matrix = styled_matrix.set_table_styles([
    # Стили для всех ячеек с данными и боковых заголовков строк
    {'selector': 'th, td', 'props': [
        ('font-size', '9px'),           # <-- Уменьшили размер текста (было 12px)
        ('text-align', 'center'),       # <-- Выравнивание по центру (горизонтально)
        ('vertical-align', 'middle'),   # <-- Выравнивание по центру (вертикально)
        ('min-width', '45px'),          # Немного уменьшили ячейки под мелкий шрифт
        ('max-width', '45px'),
        ('height', '45px'),
        ('border', '1px solid #ffffff') # Белая сетка
    ]},
    # Особые стили для верхних заголовков (чтобы они тоже были по центру и мелкими)
    {'selector': 'th.col_heading', 'props': [
        ('font-size', '9px'),           # <-- Мелкий шрифт для верхних названий
        ('text-align', 'center'),       # <-- Центрирование верхних названий
        ('transform', 'rotate(0 deg)'), # Поворот для экономии места
        ('height', '80px'),
        ('vertical-align', 'bottom')
    ]}
])

styled_matrix

,client_id,target,age,job-status,retrirement-status,gender,children,dependants,education,marital-status,industry,title,tp-state,tp-foreign,job-type,family-income,personal-income,registration-region,fact-region,postal-region,last-loan-region,region,reg-and-fact-equality,post-and-fact-equality,reg-and-post-equality,reg-fact-and-post-equality,reg-fact-post-and-last-credit-equality,realty,cars,rus-auto,dacha,cottage,garage,land,last-loan-amount,term,first-payment,state-pension-fund,residence-time,work-time,fact-phone,reg-phone,work-phone,loans,paid-loans,total-payments,total-of-delinquencies,max-delinquency-no,mean-delinquency-amount,max-delinquency-amount,previous-cards
client_id,1.0000,0.0031,0.0000,0.0114,0.0019,0.0148,0.0000,0.0127,0.0089,0.0000,0.0203,0.0000,0.0306,0.0080,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0073,0.0000,0.0000,0.0155,0.0000,0.0327,0.0303,0.0091,0.0000,0.0158,0.0144,0.0000,0.0000,0.0216,0.0000,0.0000,0.0000,0.0000,0.0136,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0152,0.0154,0.0000
target,0.0031,1.0000,0.0949,0.0000,0.0456,0.0218,0.0270,0.0545,0.0463,0.0228,0.0699,0.0447,0.0421,0.0000,0.0000,0.0784,0.0000,0.1256,0.1233,0.1270,0.1186,0.0714,0.0473,0.0000,0.0569,0.0466,0.0496,0.0248,0.0089,0.0000,0.0000,0.0000,0.0047,0.0000,0.0247,0.0666,0.0328,0.0223,0.0207,0.0000,0.0599,0.0000,0.0000,0.0254,0.0489,0.0326,0.0515,0.1092,0.0440,0.0432,0.0498
age,0.0000,0.0949,1.0000,0.0932,0.7051,0.1429,0.2496,0.3975,0.2062,0.3990,0.1630,0.0941,0.1144,0.0000,0.0641,0.0696,0.0000,0.0885,0.0891,0.0889,0.0875,0.0470,0.2874,0.1795,0.2106,0.2872,0.0146,0.1988,0.0636,0.0506,0.1557,0.0623,0.0625,0.0690,0.1332,0.0441,0.0591,0.0379,0.0000,0.0711,0.1366,0.0292,0.0599,0.0794,0.0868,0.0896,0.0354,0.0486,0.0328,0.0316,0.0496
job-status,0.0114,0.0000,0.0932,1.0000,0.1956,0.0051,0.0000,0.0255,0.0241,0.0253,0.0000,0.0110,0.0000,0.0000,0.0381,0.0170,0.0000,0.0253,0.0290,0.0284,0.0311,0.0000,0.0140,0.0000,0.0075,0.0141,0.0000,0.0219,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0102,0.4967,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0091
retrirement-status,0.0019,0.0456,0.7051,0.1956,1.0000,0.1311,0.0992,0.2377,0.0599,0.1164,0.0964,0.0374,0.0491,0.0000,0.0550,0.0113,0.0000,0.1216,0.1195,0.1219,0.1213,0.0685,0.0926,0.0617,0.0649,0.0930,0.0000,0.1238,0.0182,0.0443,0.0195,0.0450,0.0246,0.0200,0.1017,0.0397,0.0581,0.0618,0.0000,0.0454,0.0630,0.0000,0.0655,0.0395,0.0294,0.0476,0.0000,0.0211,0.0282,0.0288,0.0021
gender,0.0148,0.0218,0.1429,0.0051,0.1311,1.0000,0.0413,0.0838,0.0670,0.1522,0.4750,0.1662,0.1154,0.0167,0.2308,0.1656,0.0000,0.0706,0.0722,0.0722,0.0813,0.0309,0.0977,0.0428,0.0834,0.0977,0.0125,0.1190,0.1700,0.3390,0.0000,0.0370,0.0615,0.0222,0.0220,0.0188,0.0157,0.1876,0.0000,0.0000,0.0513,0.0108,0.0000,0.0294,0.0215,0.0302,0.0000,0.0257,0.0000,0.0000,0.0069
children,0.0000,0.0270,0.2496,0.0000,0.0992,0.0413,1.0000,0.7251,0.1050,0.2163,0.1030,0.0415,0.0529,0.0059,0.0359,0.0376,0.0000,0.1981,0.1993,0.1994,0.1880,0.0980,0.0883,0.0595,0.0615,0.0886,0.0499,0.0420,0.0000,0.0197,0.1423,0.0294,0.0301,0.0684,0.0272,0.0416,0.0000,0.0200,0.0000,0.0000,0.0199,0.0165,0.0000,0.0590,0.0420,0.0629,0.0745,0.0000,0.0000,0.0000,0.0000
dependants,0.0127,0.0545,0.3975,0.0255,0.2377,0.0838,0.7251,1.0000,0.0485,0.2579,0.0739,0.0310,0.0281,0.0000,0.0355,0.0926,0.0000,0.1201,0.1223,0.1219,0.1203,0.0596,0.0877,0.0588,0.0628,0.0885,0.0529,0.0823,0.0561,0.0651,0.1091,0.0124,0.0000,0.0655,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0637,0.0278,0.0000,0.0679,0.0631,0.0643,0.0000,0.0000,0.0000,0.0000,0.0376
education,0.0089,0.0463,0.2062,0.0241,0.0599,0.0670,0.1050,0.0485,1.0000,0.0893,0.2774,0.2818,0.0846,0.0000,0.1682,0.1805,0.0000,0.1798,0.1833,0.1831,0.1844,0.0844,0.0656,0.0375,0.0744,0.0655,0.0052,0.0566,0.0564,0.0458,0.1053,0.0319,0.0380,0.0466,0.0720,0.0419,0.0612,0.0561,0.0000,0.0000,0.1008,0.0577,0.0000,0.2971,0.2653,0.2684,0.0594,0.0580,0.0352,0.0303,0.1229
marital-status,0.0000,0

In [ ]:
# 0 ЗДЕСЬ ЗАПИСЫВАЕМ СТОЛБЦЫ, КОТОРЫЕ ЯВЛЯЮТСЯ ШУМОМ ДЛЯ ТАРГЕТА
corr_with_target = phik_matrix[phik_matrix['target'] < 0.02]['target'].sort_values().sort_values(ascending=False).reset_index()['index'].values

# 1. Вытаскиваем связь каждого признака с целевой переменной из вашей phik_matrix
target_corr = phik_matrix['target'].abs().drop('target', errors='ignore')
# 2. Получаем только чистую матрицу признаков (без target)
matrix_no_target = phik_matrix.drop(index='target', columns='target', errors='ignore').abs()
# 3. Находим пары с корреляцией > 0.70 (верхний треугольник)
upper_tri = matrix_no_target.where(np.triu(np.ones(matrix_no_target.shape), k=1).astype(bool))
pairs = upper_tri.stack()
strong_pairs = pairs[pairs > 0.8].index.tolist() # получаем список кортежей (Признак_1, Признак_2)
# 4. Проходим по парам и выбираем, кого удалить
to_drop = set()
for col1, col2 in strong_pairs:
    # Если один из признаков уже помечен на удаление, пропускаем шаг
    if col1 in to_drop or col2 in to_drop:
        continue
    # Сравниваем их полезность для target
    score1 = target_corr.get(col1, 0)
    score2 = target_corr.get(col2, 0)
    # Удаляем тот, у которого связь с таргетом слабее
    if score1 < score2:
        to_drop.add(col1)
    else:
        to_drop.add(col2)
print(f"К удалению: корреляция между переменными ({len(to_drop)} шт) между переменной и таргетом ({len(corr_with_target)} шт)")

К удалению: корреляция между переменными (0 шт) между переменной и таргетом (0 шт)


In [ ]:
to_drop_by_corr = list(to_drop) + phik_matrix[phik_matrix['target'] < 0.02]['target'].sort_values().sort_values(ascending=False).reset_index()['index'].values.tolist()
print(len(to_drop_by_corr))

0


In [ ]:
for col in to_drop_by_corr:
    del df_train[col]
print(f"Количество полных дубликатов: {df_train.duplicated().sum()}")

Количество полных дубликатов: 0


In [ ]:
phik_matrix = df_train[~df_train.isna().any(axis=1)].phik_matrix()

styled_matrix = phik_matrix.style.background_gradient(cmap='coolwarm', vmin=0, vmax=1).format("{:.4f}")

# Настраиваем стили: выравнивание и уменьшенный размер текста
styled_matrix = styled_matrix.set_table_styles([
    # Стили для всех ячеек с данными и боковых заголовков строк
    {'selector': 'th, td', 'props': [
        ('font-size', '9px'),           # <-- Уменьшили размер текста (было 12px)
        ('text-align', 'center'),       # <-- Выравнивание по центру (горизонтально)
        ('vertical-align', 'middle'),   # <-- Выравнивание по центру (вертикально)
        ('min-width', '45px'),          # Немного уменьшили ячейки под мелкий шрифт
        ('max-width', '45px'),
        ('height', '45px'),
        ('border', '1px solid #ffffff') # Белая сетка
    ]},
    # Особые стили для верхних заголовков (чтобы они тоже были по центру и мелкими)
    {'selector': 'th.col_heading', 'props': [
        ('font-size', '9px'),           # <-- Мелкий шрифт для верхних названий
        ('text-align', 'center'),       # <-- Центрирование верхних названий
        ('transform', 'rotate(0 deg)'), # Поворот для экономии места
        ('height', '80px'),
        ('vertical-align', 'bottom')
    ]}
])

styled_matrix

interval columns not set, guessing: ['target', 'age', 'retrirement-status', 'gender', 'children', 'dependants', 'family-income', 'reg-and-post-equality', 'last-loan-amount', 'term', 'first-payment', 'residence-time', 'fact-phone', 'paid-loans', 'total-payments', 'total-of-delinquencies', 'max-delinquency-no', 'mean-delinquency-amount', 'previous-cards']


,target,age,retrirement-status,gender,children,dependants,education,marital-status,industry,title,tp-state,family-income,postal-region,reg-and-post-equality,last-loan-amount,term,first-payment,residence-time,fact-phone,paid-loans,total-payments,total-of-delinquencies,max-delinquency-no,mean-delinquency-amount,previous-cards
target,1.0000,0.0968,0.0482,0.0218,0.0264,0.0520,0.0467,0.0243,0.0745,0.0432,0.0444,0.0972,0.1518,0.0656,0.0277,0.0650,0.0356,0.0201,0.0548,0.0519,0.0351,0.0504,0.1073,0.0426,0.0478
age,0.0968,1.0000,0.7003,0.1440,0.2491,0.3960,0.2059,0.4000,0.1655,0.0942,0.1164,0.0656,0.0894,0.2116,0.1314,0.0441,0.0600,0.0000,0.1358,0.0863,0.0889,0.0376,0.0489,0.0326,0.0493
retrirement-status,0.0482,0.7003,1.0000,0.1300,0.1001,0.2350,0.0606,0.1168,0.0979,0.0378,0.0506,0.0145,0.1230,0.0676,0.0975,0.0389,0.0599,0.0000,0.0610,0.0325,0.0489,0.0045,0.0205,0.0281,0.0025
gender,0.0218,0.1440,0.1300,1.0000,0.0419,0.0819,0.0648,0.1524,0.4726,0.1643,0.1156,0.1658,0.0719,0.0859,0.0217,0.0206,0.0154,0.0000,0.0511,0.0215,0.0315,0.0000,0.0243,0.0000,0.0066
children,0.0264,0.2491,0.1001,0.0419,1.0000,0.7248,0.1042,0.2164,0.1036,0.0423,0.0529,0.0370,0.2006,0.0626,0.0264,0.0410,0.0000,0.0000,0.0202,0.0427,0.0639,0.0744,0.0000,0.0000,0.0000
dependants,0.0520,0.3960,0.2350,0.0819,0.7248,1.0000,0.0485,0.2589,0.0740,0.0315,0.0286,0.0890,0.1247,0.0595,0.0000,0.0000,0.0000,0.0000,0.0632,0.0632,0.0646,0.0000,0.0000,0.0000,0.0381
education,0.0467,0.2059,0.0606,0.0648,0.1042,0.0485,1.0000,0.0912,0.2762,0.2817,0.0827,0.1829,0.1896,0.0762,0.0740,0.0407,0.0604,0.0000,0.1017,0.2648,0.2675,0.0586,0.0581,0.0311,0.1227
marital-status,0.0243,0.4000,0.1168,0.1524,0.2164,0.2589,0.0912,1.0000,0.1151,0.0669,0.0801,0.1833,0.1597,0.0326,0.0637,0.0555,0.0100,0.0000,0.0255,0.0282,0.0320,0.0523,0.0109,0.0000,0.0125
industry,0.0745,0.1655,0.0979,0.4726,0.1036,0.0740,0.2762,0.1151,1.0000,0.3500,0.6559,0.2358,0.3399,0.0821,0.0563,0.0498,0.0492,0.0000,0.0975,0.1800,0.1797,0.0000,0.0193,0.1084,0.0529
title,0.0432,0.0942,0.0378,0.1643,0.0423,0.0315,0.2817,0.0669,0.3500,1.0000,0.3131,0.2852,0.2397,0.0601,0.1148,0.0550,0.0806,0.0000,0.1377,0.2083,0.2006,0.0000,0.0180,0.0529,0.1183


In [ ]:
df_train.info(memory_usage='deep', show_counts=True, verbose=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25101 entries, 0 to 25100
Columns: 25 entries, target to previous-cards
dtypes: category(6), float64(4), int64(14), int8(1)
memory usage: 3.6 MB


In [ ]:
categorical_cols = df_train.select_dtypes(include=['object', 'category']).columns
numeric_cols = df_train.select_dtypes(include=['int8', 'int64', 'float']).columns
print(f'Категориальные ({len(categorical_cols)}) + Числовые ({len(numeric_cols)}) = Всего ({len(categorical_cols) + len(numeric_cols)})')

Категориальные (6) + Числовые (19) = Всего (25)


In [ ]:
print(f"Количество полных дубликатов: {df_train.duplicated().sum()}")
# Подсчет пропущенных значений в процентах по каждой колонке
missing_values = df_train.isnull().sum()
missing_percent = (missing_values / len(df_train)) * 100

# Объединяем в красивую таблицу (выводим только те колонки, где есть пропуски)
missing_df = pd.DataFrame({'Пропуски (шт)': missing_values, 'Пропуски (%)': missing_percent})
missing_df[missing_df['Пропуски (шт)'] > 0].sort_values(by='Пропуски (%)', ascending=False)

Количество полных дубликатов: 0


,Пропуски (шт),Пропуски (%)


In [ ]:
print(f'{df_train.shape[0]} - 2259 = {df_train.shape[0] - 2259}')
df_train.dropna(inplace=True)
print(df_train.shape[0])

In [ ]:
df_train.head(5)

,target,age,retrirement-status,gender,children,dependants,education,marital-status,industry,title,...,term,first-payment,residence-time,fact-phone,paid-loans,total-payments,total-of-delinquencies,max-delinquency-no,mean-delinquency-amount,previous-cards
0,0,41,0,1,3,1,Среднее специальное,Состою в браке,Развлечения/Искусство,Работник сферы услуг,...,12,4000.0,144,0,0,3,0,0,0.0,0.0
1,0,29,0,1,1,0,Высшее,Разведен(а),Банк/Финансы,Высококвалифиц. специалист,...,12,14000.0,72,1,0,4,0,0,0.0,0.0
2,1,27,0,1,0,0,Среднее специальное,Не состоял в браке,Транспорт,Рабочий,...,10,670.0,95,1,0,8,0,0,0.0,0.0
3,0,26,0,1,0,0,Среднее специальное,Гражданский брак,Информационные услуги,Специалист,...,6,2000.0,36,1,2,7,0,0,0.0,0.0
5,1,34,0,0,2,2,Среднее,Состою в браке,Другие сферы,Специалист,...,6,1207.0,79,1,2,13,0,0,0.0,0.0


In [ ]:
cat_summary = pd.DataFrame({
    'Уникальных': df_train[categorical_cols].nunique(),
    'Пропусков': df_train[categorical_cols].isna().sum()
})
cat_summary.sort_values(by="Уникальных")

,Уникальных,Пропусков
marital-status,5,0
tp-state,5,0
education,7,0
title,12,0
industry,31,0
postal-region,81,0


In [ ]:
phik_matrix = df_train[~df_train.isna().any(axis=1)][list(categorical_cols) + ['target']].phik_matrix()

styled_matrix = phik_matrix.style.background_gradient(cmap='coolwarm', vmin=0, vmax=1).format("{:.4f}")

# Настраиваем стили: выравнивание и уменьшенный размер текста
styled_matrix = styled_matrix.set_table_styles([
    # Стили для всех ячеек с данными и боковых заголовков строк
    {'selector': 'th, td', 'props': [
        ('font-size', '11px'),           # <-- Уменьшили размер текста (было 12px)
        ('text-align', 'center'),       # <-- Выравнивание по центру (горизонтально)
        ('vertical-align', 'middle'),   # <-- Выравнивание по центру (вертикально)
        ('min-width', '50px'),          # Немного уменьшили ячейки под мелкий шрифт
        ('max-width', '50px'),
        ('height', '50px'),
        ('border', '1px solid #ffffff') # Белая сетка
    ]},
    # Особые стили для верхних заголовков (чтобы они тоже были по центру и мелкими)
    {'selector': 'th.col_heading', 'props': [
        ('font-size', '11px'),           # <-- Мелкий шрифт для верхних названий
        ('text-align', 'center'),       # <-- Центрирование верхних названий
        ('transform', 'rotate(0 deg)'), # Поворот для экономии места
        ('height', '80px'),
        ('vertical-align', 'bottom')
    ]}
])

styled_matrix

interval columns not set, guessing: ['target']


,education,marital-status,industry,title,tp-state,postal-region,target
education,1.0000,0.0912,0.2762,0.2817,0.0827,0.1896,0.0467
marital-status,0.0912,1.0000,0.1151,0.0669,0.0801,0.1597,0.0243
industry,0.2762,0.1151,1.0000,0.3500,0.6559,0.3399,0.0745
title,0.2817,0.0669,0.3500,1.0000,0.3131,0.2397,0.0432
tp-state,0.0827,0.0801,0.6559,0.3131,1.0000,0.2007,0.0444
postal-region,0.1896,0.1597,0.3399,0.2397,0.2007,1.0000,0.1518
target,0.0467,0.0243,0.0745,0.0432,0.0444,0.1518,1.0000
